In [ ]:
import cv2
import numpy as np

class ObjectTracker:
    def __init__(self):
        self.cap = cv2.VideoCapture(0)
        self.scaling_factor = 0.5

        self.selection = None
        self.drag_start = None
        self.tracking_state = 0
        self.track_window = None
        self.hist = None

        self.window_name = "Object Tracker"
        cv2.namedWindow(self.window_name)
        cv2.setMouseCallback(self.window_name, self.mouse_event)

    def mouse_event(self, event, x, y, flags, param):
        x, y = np.int16([x, y])

        if event == cv2.EVENT_LBUTTONDOWN:
            self.drag_start = (x, y)
            self.tracking_state = 0

        if self.drag_start:
            if flags & cv2.EVENT_FLAG_LBUTTON:
                x0, y0 = self.drag_start
                x1, y1 = x, y
                self.selection = (
                    min(x0, x1),
                    min(y0, y1),
                    abs(x1 - x0),
                    abs(y1 - y0)
                )

        if event == cv2.EVENT_LBUTTONUP:
            self.drag_start = None
            self.tracking_state = 1

    def start_tracking(self):
        while True:
            ret, frame = self.cap.read()
            if not ret:
                break

            frame = cv2.resize(
                frame, None,
                fx=self.scaling_factor,
                fy=self.scaling_factor,
                interpolation=cv2.INTER_AREA
            )

            vis = frame.copy()
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

            mask = cv2.inRange(
                hsv,
                np.array((0., 30., 32.)),
                np.array((180., 255., 255.))
            )

            if self.selection:
                x, y, w, h = self.selection

                if w > 0 and h > 0:
                    hsv_roi = hsv[y:y+h, x:x+w]
                    mask_roi = mask[y:y+h, x:x+w]

                    hist = cv2.calcHist([hsv_roi], [0], mask_roi, [16], [0, 180])
                    cv2.normalize(hist, hist, 0, 255, cv2.NORM_MINMAX)

                    self.track_window = (x, y, w, h)
                    self.hist = hist
                    self.selection = None

            if self.tracking_state == 1 and self.hist is not None:
                back_proj = cv2.calcBackProject([hsv], [0], self.hist, [0, 180], 1)
                back_proj &= mask

                term_crit = (
                    cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
                    10,
                    1
                )

                track_box, self.track_window = cv2.CamShift(
                    back_proj,
                    self.track_window,
                    term_crit
                )

                cv2.ellipse(vis, track_box, (0, 255, 0), 2)

            cv2.imshow(self.window_name, vis)

            if cv2.waitKey(1) == 27:
                break

        self.cap.release()
        cv2.destroyAllWindows()


tracker = ObjectTracker()
tracker.start_tracking()